# wavexplain — run it on your own data

Upload your sales data, train a forecasting model on it, and get a card for each product that shows **why** the forecast is what it is: how much comes from the typical pattern, an active promotion, and the recent trend.

**How to use this:** run each cell top to bottom (Shift+Enter). First turn on the free GPU: **Runtime → Change runtime type → GPU → Save**.

**What this is:** a research tool. The breakdown shows how the model responds to your data (counterfactual), not a proven real-world cause. It trains a small model in a few minutes and has not been validated at a real store, so treat the output as a decision aid, not a guarantee.


### 1. Setup
Install the wavexplain package and check the GPU.

In [ ]:
!pip install -q git+https://github.com/kesjien/wavexplain.git
!pip install -q pandas

import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import OrderedDict
from wavexplain import MultiSeriesWaveNet, CounterfactualExplainer
print('wavexplain loaded | torch', torch.__version__, '| GPU:', torch.cuda.is_available())


### 2. Your data
Your CSV needs at least a **date**, an **item/product id**, and a **sales** number per row. A **store** column and a **promotion** flag (0/1) are optional but make the cards better.

Leave `USE_SAMPLE = True` to try it with built-in demo data first. Set it to `False` to upload your own CSV.

In [ ]:
USE_SAMPLE = True   # set False to upload your own CSV

if USE_SAMPLE:
    rng = np.random.default_rng(0)
    dates = pd.date_range('2016-01-01', periods=400, freq='D')
    rows = []
    for store in [1, 2]:
        for item in [101, 102, 103]:
            base = rng.integers(5, 40)
            weekly = 1 + 0.4*np.sin(np.arange(len(dates))*2*np.pi/7)
            promo = (rng.random(len(dates)) < 0.08).astype(int)
            sales = base*weekly*(1 + 0.8*promo) + rng.normal(0, base*0.1, len(dates))
            sales = np.clip(np.round(sales), 0, None).astype(int)
            for dt, s, p in zip(dates, sales, promo):
                rows.append([dt.date(), store, item, int(s), int(p)])
    df = pd.DataFrame(rows, columns=['date','store','item','sales','promotion'])
    print('Using built-in sample:', len(df), 'rows')
else:
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(list(up.keys())[0])
    print('Loaded your file:', len(df), 'rows')

df.head()


### 3. Check the columns
The code guesses which column is which. If a guess is wrong, fix it by hand, e.g. `COLS['sales'] = 'my_units_column'`, then run the next cell.

In [ ]:
def guess(cols, options):
    for o in options:
        for c in cols:
            if o in str(c).lower():
                return c
    return None

cols = list(df.columns)
COLS = {
    'date':  guess(cols, ['date','day','ds']),
    'item':  guess(cols, ['item','sku','product']),
    'store': guess(cols, ['store','shop','location']),      # optional
    'sales': guess(cols, ['sales','units','qty','quantity','demand','y']),
    'promo': guess(cols, ['promo','onpromotion','discount','deal']),  # optional
}
print('Detected columns:', COLS)
print('If any are wrong, fix them here before running the next cell.')


### 4. Clean and reshape
Builds a dense daily grid per product, fills missing days with zero, drops negatives, and produces the panels the model trains on.

In [ ]:
HORIZON = 16          # days the model forecasts
INPUT_LENGTH = 90     # days of history the model looks back on

d = df.copy()
d[COLS['date']] = pd.to_datetime(d[COLS['date']], errors='coerce')
d = d.dropna(subset=[COLS['date']])
if COLS['store'] is None:
    d['_store'] = 0; COLS['store'] = '_store'
if COLS['promo'] is None:
    d['_promo'] = 0; COLS['promo'] = '_promo'
d[COLS['sales']] = pd.to_numeric(d[COLS['sales']], errors='coerce').fillna(0).clip(lower=0)
d[COLS['promo']] = (pd.to_numeric(d[COLS['promo']], errors='coerce').fillna(0) > 0).astype(int)

full_dates = pd.date_range(d[COLS['date']].min(), d[COLS['date']].max(), freq='D')
num_days = len(full_dates)
if num_days < INPUT_LENGTH + HORIZON + 10:
    INPUT_LENGTH = max(28, num_days - HORIZON - 10)
    print('Short history: reduced INPUT_LENGTH to', INPUT_LENGTH)
assert num_days >= INPUT_LENGTH + HORIZON, 'Need more history: at least ~4 months per product.'

def as_int(x, fb):
    try: return int(x)
    except Exception: return fb

keys = d[[COLS['store'], COLS['item']]].drop_duplicates().values.tolist()
sales_rows, promo_rows, index_rows = [], [], []
for sid, (store, item) in enumerate(keys):
    g = d[(d[COLS['store']]==store) & (d[COLS['item']]==item)]
    g = g.groupby(COLS['date']).agg({COLS['sales']:'sum', COLS['promo']:'max'}).reindex(full_dates, fill_value=0)
    sales_rows.append(g[COLS['sales']].to_numpy(dtype='float32'))
    promo_rows.append(g[COLS['promo']].to_numpy(dtype='float32'))
    index_rows.append({'series_id': sid, 'store_nbr': as_int(store, sid), 'item_nbr': as_int(item, sid)})

sales_panel = np.vstack(sales_rows)
promo_panel = np.vstack(promo_rows)
series_index = pd.DataFrame(index_rows)
print(f'{sales_panel.shape[0]} products, {num_days} days ({full_dates.min().date()} to {full_dates.max().date()}).')
print(f'Promotion days present: {int(promo_panel.sum())}.')


### 5. Train on your data
Trains a fresh model on your products and reports validation error on the held-out last 16 days. `QUICK = True` is fast; set it to `False` for a more thorough run.

The score is RMSLE (lower is better). Uploaded data has no perishable flag, so this is the equal-weight version of the competition's NWRMSLE, same formula.

In [ ]:
QUICK = True
NUM_EPOCHS = 8 if QUICK else 30
SAMPLES_PER_EPOCH = 20_000 if QUICK else 60_000
BATCH = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
L, H, D = INPUT_LENGTH, HORIZON, sales_panel.shape[1]

class WindowDataset(Dataset):
    def __init__(self, mode):
        self.mode = mode
    def __len__(self):
        return SAMPLES_PER_EPOCH if self.mode == 'train' else sales_panel.shape[0]
    def __getitem__(self, idx):
        if self.mode == 'train':
            s = np.random.randint(0, sales_panel.shape[0])
            start = np.random.randint(0, D - L - H + 1)
        else:
            s, start = idx, D - L - H
        sw = sales_panel[s, start:start+L]; pw = promo_panel[s, start:start+L]
        tgt = sales_panel[s, start+L:start+L+H]
        x = torch.from_numpy(np.stack([sw, pw]).copy()).float()   # raw; log applied in loop
        return x, torch.tensor(s, dtype=torch.long), torch.from_numpy(tgt.copy()).float()

train_loader = DataLoader(WindowDataset('train'), batch_size=BATCH)
val_loader   = DataLoader(WindowDataset('val'),   batch_size=BATCH)

def rmsle(y_true, y_pred):
    y_true = np.clip(y_true, 0, None); y_pred = np.clip(y_pred, 0, None)
    return float(np.sqrt(((np.log1p(y_pred) - np.log1p(y_true))**2).mean()))

model = MultiSeriesWaveNet(num_series=sales_panel.shape[0], horizon=H, num_covariates=1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

best = None
for ep in range(1, NUM_EPOCHS+1):
    model.train(); run = 0.0; nb = 0
    for x, sid, tgt in train_loader:
        x, sid, tgt = x.to(device), sid.to(device), tgt.to(device)
        log_in = torch.cat([torch.log1p(x[:,0:1].clamp(min=0)), x[:,1:2]], dim=1)
        opt.zero_grad()
        loss = loss_fn(model(log_in, sid), torch.log1p(tgt.clamp(min=0)))
        loss.backward(); opt.step(); run += loss.item(); nb += 1
    model.eval(); tr, pr = [], []
    with torch.no_grad():
        for x, sid, tgt in val_loader:
            x, sid = x.to(device), sid.to(device)
            log_in = torch.cat([torch.log1p(x[:,0:1].clamp(min=0)), x[:,1:2]], dim=1)
            pred = torch.expm1(model(log_in, sid).clamp(min=0)).cpu().numpy()
            pr.append(pred); tr.append(tgt.numpy())
    val = rmsle(np.concatenate(tr), np.concatenate(pr))
    print(f'epoch {ep}/{NUM_EPOCHS}  train MSE {run/nb:.4f}  val RMSLE {val:.4f}')
    if best is None or val < best:
        best = val; torch.save(model.state_dict(), 'user_model.pt')
model.eval()
print(f'done. best val RMSLE {best:.4f} (lower is better)')


### 6. Make the cards
Uses the wavexplain `CounterfactualExplainer` to decompose each forecast, then writes `cards.json` (for the demo page) and `dashboard.html`, and shows them below.

In [ ]:
REPORT_HORIZON = HORIZON   # sum this many days for the headline + buckets
HISTORY_PLOTTED = 16
RECENT_DAYS = 14

def build_card(series_id):
    start = D - L - H
    sales_w = sales_panel[series_id, start:start+L].astype(float)
    promo_w = promo_panel[series_id, start:start+L].astype(float)
    log_sales = np.log1p(np.clip(sales_w, 0, None))
    full_input = np.stack([log_sales, promo_w])
    baseline_values = [float(log_sales.mean()), 0.0]

    promo_mask = promo_w > 0.5
    recent_mask = np.zeros(L, bool); recent_mask[-RECENT_DAYS:] = True
    typical_mask = (~promo_mask) & (~recent_mask)
    recent_np = recent_mask & (~promo_mask)
    reveal = OrderedDict([('seasonal_pattern', typical_mask), ('recent_trend', recent_np), ('promotion_effect', promo_mask)])

    out_tf = lambda t: torch.expm1(t.clamp(min=0))[:, :REPORT_HORIZON].sum(dim=1, keepdim=True)
    ex = CounterfactualExplainer(model, series_id=series_id, device=device, output_transform=out_tf)
    contribs, base_pred, full_pred = ex.explain(full_input, baseline_values, reveal, output_index=0)
    typical = base_pred + contribs['seasonal_pattern']; trend = contribs['recent_trend']; promo = contribs['promotion_effect']

    x = torch.tensor(full_input[None], dtype=torch.float32, device=device)
    sid_t = torch.tensor([series_id], dtype=torch.long, device=device)
    with torch.no_grad():
        per_day = torch.expm1(model(x, sid_t).clamp(min=0))[0].cpu().numpy()[:REPORT_HORIZON]

    promotion_driven = abs(promo) >= max(abs(trend), 1.0) and promo > 0.1*max(full_pred, 1.0)
    if promotion_driven: expl = 'Mostly explained by the active promotion, not a shift in underlying demand.'
    elif trend > max(abs(typical)*0.15, 1): expl = 'Rising demand. Most of the forecast comes from a recent upward trend, not a promotion.'
    elif trend < -max(abs(typical)*0.15, 1): expl = 'Softening demand. The forecast is below the usual pattern because of a recent downward trend.'
    else: expl = "Follows this product's typical sales pattern."
    row = series_index[series_index['series_id']==series_id].iloc[0]
    return {'store': int(row['store_nbr']), 'item': int(row['item_nbr']), 'promotion_driven': bool(promotion_driven),
            'horizon_label': f'next {REPORT_HORIZON} days', 'total_forecast': round(float(full_pred)), 'explanation': expl,
            'drivers': [{'label':'Typical pattern for this product','value':round(float(typical)),'kind':'base'},
                        {'label':'Active promotion','value':round(float(promo)),'kind':'promo'},
                        {'label':'Recent trend','value':round(float(trend)),'kind':'trend'}],
            'history': [round(float(v),1) for v in sales_w[-HISTORY_PLOTTED:]],
            'forecast': [round(float(v),1) for v in per_day]}

import json
series_ids = list(series_index['series_id'])[:20]
cards = [build_card(s) for s in series_ids]
json.dump(cards, open('cards.json','w'), indent=2)
print('wrote cards.json with', len(cards), 'cards')


### 7. See the cards
Renders the cards inline (same look as the demo page).

In [ ]:
def chart_svg(history, forecast):
    W,Hh,xL,xR,yT,yB = 500,150,40,470,20,120
    h,n = len(history), len(history)+len(forecast)
    allv = history+forecast; lo,hi = min(allv),max(allv)
    if hi==lo: hi+=1; lo-=1
    xa = lambda i: xL+i*(xR-xL)/(n-1); ya = lambda v: yB-(v-lo)/(hi-lo)*(yB-yT)
    hp = ' '.join(f'{xa(i):.1f},{ya(v):.1f}' for i,v in enumerate(history))
    fi = [h-1]+[h+k for k in range(len(forecast))]; fv=[history[-1]]+forecast
    fp = ' '.join(f'{xa(i):.1f},{ya(v):.1f}' for i,v in zip(fi,fv))
    dv = (xa(h-1)+xa(h))/2
    return (f'<svg viewBox="0 0 {W} {Hh}" style="width:100%"><line x1="{dv:.1f}" y1="16" x2="{dv:.1f}" y2="124" stroke="#c9c7bd" stroke-dasharray="3 3"/>'
            f'<polyline fill="none" stroke="#888780" stroke-width="1.5" points="{hp}"/>'
            f'<polyline fill="none" stroke="#EF9F27" stroke-width="2" stroke-dasharray="5 4" points="{fp}"/></svg>')

def card_html(c):
    badge = ('#FAEEDA','#854F0B','Promotion-driven') if c['promotion_driven'] else ('#F1EFE8','#5F5E5A','Standard pattern')
    mx = max(abs(x['value']) for x in c['drivers']) or 1
    rows = ''
    for dvr in c['drivers']:
        col = '#EF9F27' if dvr['kind']=='promo' else '#888780'
        val = dvr['value'] if dvr['kind']=='base' else (f"+{dvr['value']}" if dvr['value']>=0 else dvr['value'])
        rows += (f"<div style='display:flex;justify-content:space-between;font-size:13px;margin:8px 0 4px'><span style='color:#5f5e5a'>{dvr['label']}</span><span>{val}</span></div>"
                 f"<div style='background:#f1efe8;border-radius:4px;height:8px'><div style='width:{max(2,round(abs(dvr['value'])/mx*100))}%;height:100%;border-radius:4px;background:{col}'></div></div>")
    return (f"<div style='background:#fff;border:1px solid #e0ded4;border-radius:12px;padding:20px 24px;max-width:420px;margin:16px auto;font-family:-apple-system,Arial,sans-serif'>"
            f"<div style='display:flex;justify-content:space-between'><span style='color:#5f5e5a;font-size:13px'>Store {c['store']} &middot; Item {c['item']}</span>"
            f"<span style='background:{badge[0]};color:{badge[1]};font-size:12px;padding:3px 10px;border-radius:6px;font-weight:600'>{badge[2]}</span></div>"
            f"<p style='color:#5f5e5a;font-size:13px;margin:12px 0 2px'>Predicted sales, {c['horizon_label']}</p>"
            f"<p style='font-size:32px;font-weight:600;margin:0'>{c['total_forecast']} units</p>{chart_svg(c['history'],c['forecast'])}"
            f"<p style='font-size:14px;color:#444;margin:8px 0 12px'>{c['explanation']}</p>{rows}</div>")

from IPython.display import HTML, display
display(HTML(''.join(card_html(c) for c in cards)))


### 8. Download
Saves `cards.json` to your computer. Drop it into the `wavexplain` repo (next to `index.html`) to update the live demo at your GitHub Pages URL.

In [ ]:
from google.colab import files
files.download('cards.json')


---
**Honest limits.** This trains a small model on your data in a few minutes; it is not production-hardened or validated at scale. Longer forecasts (toward day 16) are less accurate than short ones. The promotion bucket is the least-tested part of the breakdown, so treat a 'Promotion-driven' label as a hint to check, not a proven fact. The card shows the model's own response to your data, not real-world causation.
